In [1]:
import pandas as pd
import numpy as np

# ==========================
# LOAD DATA
# ==========================

file="../LARPDatabase_2015.csv"

df=pd.read_csv(
    file,
    dtype=str
)

print("Original Shape:",df.shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

df.columns=(
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n"," ",regex=False)
    .str.replace(r"\s+","_",regex=True)
)

# ==========================
# STANDARDIZE EMPTY VALUES
# ==========================

empty_values=[

    "",
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "nan"

]

df=df.replace(
    empty_values,
    np.nan
)

# ==========================
# HANDLE SERIAL COLUMN
# ==========================

si_col=None

for col in df.columns:

    clean=(
        str(col)
        .lower()
        .replace("_","")
        .replace(" ","")
    )

    if clean=="unnamed:0":

        si_col=col
        break

if si_col:

    df=df.rename(
        columns={
            si_col:"SI_No"
        }
    )

# ==========================
# REMOVE UNKNOWN COLUMNS
# KEEP SI_No
# ==========================

unknown=[]

for col in df.columns:

    if (
        "unnamed" in col.lower()
        and col!="SI_No"
    ):

        unknown.append(col)

df=df.drop(
    columns=unknown,
    errors="ignore"
)

print("\nUnknown Columns Removed:")
print(unknown)

# ==========================
# REMOVE 100% EMPTY COLUMNS
# ==========================

empty_cols=[]

for col in df.columns:

    is_empty=(

        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .all()

    )

    if is_empty:

        empty_cols.append(col)

df=df.drop(
    columns=empty_cols,
    errors="ignore"
)

print("\nRemoved Empty Columns:")
print(empty_cols)

# ==========================
# REMOVE DUPLICATES
# ==========================

duplicates=df.duplicated().sum()

df=df.drop_duplicates()

print("\nDuplicates Removed:",duplicates)

# ==========================
# RESET SI_No
# ==========================

df=df.reset_index(
    drop=True
)

if "SI_No" in df.columns:

    df["SI_No"]=range(
        1,
        len(df)+1
    )

else:

    df.insert(
        0,
        "SI_No",
        range(
            1,
            len(df)+1
        )
    )

print("\nSI_No reordered")

# ==========================
# MISSING SUMMARY
# ==========================

missing=(

    df
    .replace("",np.nan)
    .isna()
    .sum()

)

missing=missing[
    missing>0
]

print("\nMissing Values Summary:")
print(missing)

# ==========================
# FINAL REPORT
# ==========================

print("\nFinal Shape:")
print(df.shape)

print("\nColumns After Cleaning:")
print(list(df.columns))

# ==========================
# EXPORT
# ==========================

output="cleaned_LARPDatabase_2015.xlsx"

with pd.ExcelWriter(
    output,
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        index=False,
        sheet_name="Cleaned_Data"
    )

print("\nSaved:",output)

Original Shape: (61050, 22)

Unknown Columns Removed:
[]

Removed Empty Columns:
[]

Duplicates Removed: 0

SI_No reordered

Missing Values Summary:
MANAGEMENT                  966
VESSEL                      966
AREA_OF_CONCERN           21144
DESCRIPTION                  17
IMMEDIATE_ACTION_TAKEN     7758
WORK_EQUIPMENT            54445
LARP_PROCEDURE            34363
LARP_SAFETY_DEVICE        54339
WORK_ENVIRONMENT          56635
HOUSE_KEEPING             54574
PERMIT_TO_WORK            60379
LARP_PPE                  52753
WORK_POSITION             53365
REPORTED_BY                8957
CLOSURE_REMARK            55449
CLOSURE_DATE              55443
CLOSED_BY                 55444
dtype: int64

Final Shape:
(61050, 22)

Columns After Cleaning:
['SI_No', 'MANAGEMENT', 'VESSEL', 'REF_NO', 'OCCURRENCE_DATE', 'STATUS', 'AREA_OF_CONCERN', 'DESCRIPTION', 'IMMEDIATE_ACTION_TAKEN', 'CATEGORY', 'WORK_EQUIPMENT', 'LARP_PROCEDURE', 'LARP_SAFETY_DEVICE', 'WORK_ENVIRONMENT', 'HOUSE_KEEPING', 'PE